# Sentiment Classification Project

In [1]:
import ctypes
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libcudart.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvrtc.so", mode=ctypes.RTLD_GLOBAL)
ctypes.CDLL("/cluster/data/cuda/12.8.1/lib64/libnvJitLink.so", mode=ctypes.RTLD_GLOBAL)

import sys
sys.path.insert(0, "/home/taekim/.local/lib/python3.14/site-packages")

import torch
print(torch.__file__)
print(torch.version.cuda)
x = torch.randn(3, 3).cuda()
print(x @ x)

/home/taekim/.local/lib/python3.14/site-packages/torch/__init__.py
12.8
tensor([[ 3.2075,  1.6063, -1.8702],
        [ 1.8302,  0.8279,  1.1690],
        [ 0.1211, -0.6988,  2.6651]], device='cuda:0')


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Verify Setup
Make sure cuda (GPU) is available

In [3]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5060 Ti


# Load data

In [4]:
train_full = pd.read_csv("/cluster/courses/cil/text-classification/data/train.csv")

# Data preprocessing TODO
We need to preprocess data. Step that come to my mind:
 - Remove word count outliers. (The vibe of a review comes across after 100 words or so).
 - We have german and english data. Should we translate everything to english
 - Should we search for smiley an insert text for them.
 - Should we search for ** (bold markers) and emphasize this word differenetly? 

# Build Validation Set
Todo: 

Currently we use 90% of the reviews for training, and the remaining 10% for validation
This should be optimized. Use k-fold validation or something like that to get most of our data as training and enable proper hyperparameter tuning later on.

In [5]:
train_df, val_df = train_test_split(
        train_full, test_size=0.1, stratify=train_full["label"], random_state=42
)

# Baselines
The current baseline implementation follows the structure:
$$\text{Output} = \text{Classifier}(\text{EmbeddingForTokens}(x))$$

In this specific case:
* **EmbeddingForTokens**: They use `CountVectorizer`, which creates a "Bag of Words" representation (counting word frequencies).
* **Classifier**: They use **Logistic Regression**.

We could implement additional baselines by choosing more complex **EmbeddingForTokens** methods (such as word-level embeddings or pre-trained vectors) and more sophisticated **Classifier** models (such as Random Forests or simple Neural Networks (MLP)).

# TRAIN PIPELINE

## Overview
A modular training pipeline that evaluates combinations of (vectorizer, classifier) pairs.
Defined across three files: embeddings.py, models.py, and train_loop.py.

## Step 1 — Vectorization (embeddings.py)
For classical vectorizers (BoW, TF-IDF), the signature is:
    (X_train, X_val) -> (X_train_embedded, X_val_embedded)
The vocabulary is fit on X_train only, then applied to both splits.

For pretrained embedding models (BERT, GloVe), sentences are encoded
independently with no fitting step:
    X -> X_embedded

## Step 2 — Classification (models.py)
Each classifier is a factory function that returns a fresh, untrained model instance.
Keeping them as functions (rather than instances) ensures each combination in the
loop starts from a clean state.

model is then used to fit X_train on Y_train: model.fit(X_train, Y_train)

## Step 3 — Train Loop (train_loop.py)
Takes a list of (vectorizer_fn, model_fn) tuples. For each combination it:
  1. Vectorizes the data
  2. Trains the classifier on the training embeddings X_train, Y_train
  3. Evaluates on both train and validation splits

Returns a list of result dicts, one per combination:
    {
        "vectorizer":          str,   # name of the vectorizer function
        "model":               str,   # name of the model factory
        "training_score":      float,
        "training_mae":        float,
        "training_accuracy":   float,
        "validation_score":    float,
        "validation_mae":      float,
        "validation_accuracy": float
    }

In [6]:
# # so that it reloads the modules every time we run this cell, so we always have the latest version of the code
# %load_ext autoreload
# %autoreload 2

# import importlib
# import train_loop
# importlib.reload(train_loop)
# from train_loop import train_loop

In [7]:
# NOTE: don't do GloVe, as only for English
# downlaod GloVe embeddings (100-dimensional vectors) and extract the file
# import urllib.request
# import zipfile

# url = "http://nlp.stanford.edu/data/glove.6B.zip"
# urllib.request.urlretrieve(url, "glove.6B.zip")
# with zipfile.ZipFile("glove.6B.zip", "r") as f:
#     f.extract("glove.6B.100d.txt")  # 100-dimensional vectors

## Simple baselines

As baselines, we first evaluate these combinations and then try to beat it later with more sophisticated approaches.
1) BoW  +  Logistic Regression
2) TF-IDF  +  Logistic Regression
3) multilingual pre-trained embeddings + Logistic Regression

Notes: MLP and Random Forest take way too long on such sparse and high dimensional embeddings as BoW and TF-IDF... Skipped for now

In [8]:
# import sys
# import subprocess
# subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"])

In [9]:
import os
os.chdir('/home/taekim/Garnella')

In [10]:
%load_ext autoreload
%autoreload 2


In [11]:
import os
os.environ["PATH"] = "/cluster/courses/cil/envs/envs/text-classification/bin:" + os.environ.get("PATH", "")
import sys
print(sys.executable)

/cluster/courses/cil/envs/envs/text-classification/bin/python


In [12]:
from embeddings import *
from models import *
from train_loop_caching import train_loop

# Cell 1: Embedding comparison BASELINE
combinations_embeddings = [
    #(get_tfidf_embeddings, get_logistic_regression),
   # (get_multilingual_embeddings, get_logistic_regression),
    #(get_gemma_embeddings, get_logistic_regression),
    #(get_qwen_embeddings, get_logistic_regression),
]
#results_emb = train_loop(train_df, val_df, combinations_embeddings)
#pd.DataFrame(results_emb).sort_values("validation_score", ascending=False)

/home/taekim/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Cell 2: Model comparison GEMMA TRY
# combinations_models = [
#     (get_gemma_embeddings, get_logistic_regression),
#     (get_gemma_embeddings, get_linear_svm),
#     (get_gemma_embeddings, get_knn),
#     (get_gemma_embeddings, get_mlp),
#     (get_gemma_embeddings, get_xgboost),
# ]
# results_models = train_loop(train_df, val_df, combinations_models)
# pd.DataFrame(results_models).sort_values("validation_score", ascending=False)

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

from embeddings_adv import *

finetune_gemma(train_df.sample(10000, random_state=0),
               val_df=val_df.sample(1000, random_state=0),
               epochs=1, batch_size=128, lr=2e-4, max_seq_length=64,
               n_train_pairs=10_000, n_eval_pairs=1_000)

In [ ]:
import os
from embeddings_adv import *

# Clear stale finetuned embeddings from the smoke test
for suffix in ["_train.npy", "_val.npy"]:
    p = f"./embedding_cache/gemma_finetuned{suffix}"
    if os.path.exists(p):
        os.remove(p)

# Retrain on full dataset
finetune_gemma(train_df, val_df=val_df,
               epochs=3, batch_size=128, lr=2e-4, max_seq_length=64)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 7489.70it/s]


trainable params: 4,177,920 || all params: 307,041,024 || trainable%: 1.3607


The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


Step,Training Loss
50,0.436539
100,0.483459
150,0.486573
200,0.491075
250,0.484653
300,0.487898
350,0.489568
400,0.495526
450,0.492211
500,0.493789


KeyboardInterrupt: 

In [ ]:
from embeddings_adv import *

# # Compare embeddings + run XGBoost (both need GPU)
combinations_gpu = [
    (get_gemma_embeddings_v2,          get_logistic_regression),
    #(get_gemma_embeddings_v2,          get_xgboost),
    (get_gte_multilingual_embeddings,  get_logistic_regression),
    #(get_gte_multilingual_embeddings,  get_xgboost),
    #(get_gemma_finetuned_embeddings,   get_logistic_regression),
    #(get_gemma_finetuned_embeddings,   get_xgboost),
]

results_gpu = train_loop(train_df, val_df, combinations_gpu)
pd.DataFrame(results_gpu).sort_values("validation_score", ascending=False)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 21794.36it/s]
The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss


KeyboardInterrupt: 